In [1]:
#!wget -O opensubtitles_en-fr.zip "https://object.pouta.csc.fi/OPUS-OpenSubtitles/v2018/moses/en-fr.txt.zip"


In [2]:
#!unzip opensubtitles_en-fr.zip

In [3]:
import random
import pandas as pd

# 파일 불러오기
with open("OpenSubtitles.en-fr.en", encoding="utf-8") as f_en, \
     open("OpenSubtitles.en-fr.fr", encoding="utf-8") as f_fr:
    en_lines = f_en.readlines()
    fr_lines = f_fr.readlines()

# 무작위 10,000개만 추출
idx = random.sample(range(len(en_lines)), 10000)
df = pd.DataFrame({
    "src": [en_lines[i].strip() for i in idx],
    "tgt": [fr_lines[i].strip() for i in idx]
})
print(df.head())
print("total number of samples:", len(df))


                                                 src  \
0                                        Yes, madam.   
1                                   - Yes, yes, yes.   
2           But again, Sarah Jane has, Gavin hasn't.   
3  Yeah, you my brother, but goddamn, man. That's...   
4  It will take me years to devise the manner of ...   

                                                 tgt  
0                                       Oui, madame.  
1                                        - Oui, oui.  
2        Mais Sarah Jane n'est pas là, Gavin est là.  
3                T'es mon frère, mais bon sang, mec.  
4  Cela me prendra des années pour imaginer comme...  
total number of samples: 10000


In [4]:
df = df.dropna()
df = df[df["src"].str.strip() != ""]
df = df[df["tgt"].str.strip() != ""]

df = df.drop_duplicates(subset=["src", "tgt"])

df["src"] = df["src"].str.lower().str.strip()
df["tgt"] = df["tgt"].str.lower().str.strip()

print(df.sample(5))
print("total:", len(df))

                                                    src  \
4922  and in the meantime, ' you work for her advanc...   
7510                             your trial is at hand.   
6530                                         i did not!   
6516                                     no, i haven't.   
4447                      but not to the patients, huh?   

                                                    tgt  
4922  et en même temps, vous travaillez pour son ava...  
7510                            ton épreuve est proche.  
6530                                ce n'est pas vrai !  
6516                                             - non.  
4447                      mais pas envers les malades ?  
total: 9805


In [5]:
import re

def clean_text(text):
    text = re.sub(r"<.*?>", "", text)      # HTML 태그 제거
    text = re.sub(r"&[a-z]+;", "", text)   # HTML 엔티티 제거
    text = re.sub(r"[\(\)\[\]\{\}]", "", text)  # 괄호류 제거
    text = re.sub(r"^\W+|\W+$", "", text)  # 양끝 기호 제거
    return text.strip()

df["src"] = df["src"].apply(clean_text)
df["tgt"] = df["tgt"].apply(clean_text)

In [6]:
from sklearn.utils import shuffle
df = shuffle(df, random_state=42).reset_index(drop=True)


In [7]:
train_df = df.sample(frac=0.9, random_state=42)
valid_df = df.drop(train_df.index)


In [8]:
train_df.to_csv("train.csv", index=False)
valid_df.to_csv("valid.csv", index=False)


In [9]:
#!python3 -m pip install --user unbabel-comet



In [10]:
from transformers import MarianTokenizer, MarianMTModel
from datasets import load_metric
from comet import download_model, load_from_checkpoint
import torch
import time
import numpy as np
import pandas as pd

# 모델 이름 (proposal의 baseline)
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")

print("✅ Model loaded:", model_name)



/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


✅ Model loaded: Helsinki-NLP/opus-mt-en-fr


In [11]:
# 샘플 데이터 (이미 전처리된 DataFrame df 사용)
# 필요에 따라 500~1000개로 제한 (속도 고려)
sample_df = df.sample(500, random_state=42).reset_index(drop=True)
src_texts = sample_df["src"].tolist()
tgt_texts = sample_df["tgt"].tolist()


In [12]:
preds, latencies = [], []

for text in src_texts:
    start = time.time()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
    output = model.generate(**inputs)
    end = time.time()

    pred = tokenizer.decode(output[0], skip_special_tokens=True)
    preds.append(pred)
    latencies.append(end - start)

# 평균 latency (초 단위)
avg_latency = np.mean(latencies)
print(f"✅ Average latency per sentence: {avg_latency:.3f} seconds")


✅ Average latency per sentence: 0.270 seconds


In [13]:
metric_bleu = load_metric("sacrebleu")
metric_chrf = load_metric("chrf")

bleu = metric_bleu.compute(predictions=preds, references=[[t] for t in tgt_texts])
chrf = metric_chrf.compute(predictions=preds, references=[[t] for t in tgt_texts])

print(f"BLEU: {bleu['score']:.2f}")
print(f"chrF: {chrf['score']:.2f}")


/var/folders/00/hlc640bs3hx3jf_rqtdxfpkc0000gn/T/ipykernel_9760/2869170210.py:1: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric_bleu = load_metric("sacrebleu")


BLEU: 19.19
chrF: 47.49


In [15]:
# COMET 모델 다운로드 및 로드
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

# COMET 입력 포맷 구성
data = [{"src": s, "mt": p, "ref": t} for s, p, t in zip(src_texts, preds, tgt_texts)]

comet_score = comet_model.predict(
    data,
    batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0,
    num_workers=1,
)
print(f"COMET mean score: {np.mean(comet_score['system_score']):.4f}")


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 113359.57it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
Predicting: 0it [00:00, ?it/s]huggingface/tokenizers: The current pr

COMET mean score: 0.7417


In [16]:
# 간단한 proxy latency 측정 (offline baseline 용)
# AP = average(output_len / input_len)
# AL, DAL은 simulation 없는 baseline에서는 동일하게 취급 가능 (offline=0 lag)
input_lens = [len(tokenizer.encode(s)) for s in src_texts]
output_lens = [len(tokenizer.encode(p)) for p in preds]
avg_prop = np.mean(np.array(output_lens) / np.array(input_lens))

print(f"Average Proportion (AP): {avg_prop:.3f}")
print(f"Average Lagging (AL): ~0 (offline model)")
print(f"Differentiable AL (DAL): ~0 (offline model)")


Average Proportion (AP): 1.649
Average Lagging (AL): ~0 (offline model)
Differentiable AL (DAL): ~0 (offline model)


In [17]:
print("\n===== Evaluation Summary =====")
print(f"BLEU: {bleu['score']:.2f}")
print(f"chrF: {chrf['score']:.2f}")
print(f"COMET: {np.mean(comet_score['system_score']):.4f}")
print(f"Latency (sec/sentence): {avg_latency:.3f}")
print(f"AP: {avg_prop:.3f} | AL ≈ 0 | DAL ≈ 0")
print("==============================")



===== Evaluation Summary =====
BLEU: 19.19
chrF: 47.49
COMET: 0.7417
Latency (sec/sentence): 0.270
AP: 1.649 | AL ≈ 0 | DAL ≈ 0
